<div align="center">
  <h3><b>ESCUELA POLITÉCNICA NACIONAL</b></h3>
  <h3><b>FACULTAD DE INGENIERÍA EN SISTEMAS</b></h3>
  <h3><b>INGENIERÍA EN CIENCIAS DE LA COMPUTACIÓN</b></h3>
  <h3><b>RECUPERACIÓN DE LA INFORMACIÓN</b></h3>
</div>

---
**Nombre**   Mark Hernández        
**Fecha**    29/06/26  
**Docente**  Iván Carrera

# Ejercicio 9: Uso de la API de Google Gemini

En este ejercicio vamos a aprender a utilizar la API de OpenAI

## 1. Uso básico

El siguiente código sirve para conectarse con la API de Google Gemini de forma básica

In [2]:
import google.generativeai as genai
from dotenv import load_dotenv
import os

# Cargamos las variables del archivo .env
load_dotenv()

# Obtenemos la clave de forma segura
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# Configuramos el API Key. 
genai.configure(api_key=GOOGLE_API_KEY)

# Prueba de uso básico con el modelo flash
model = genai.GenerativeModel('gemini-3.5-flash')
response = model.generate_content("Explica en una oración qué es la recuperación de la información.")

print("Respuesta del modelo:")
print(response.text)

Respuesta del modelo:
La recuperación de la información es el proceso de buscar, organizar y extraer datos o documentos relevantes de una gran colección para satisfacer una necesidad o consulta específica de un usuario.


## 2. Retrieval

### 2.1 Cargo el corpus de 20 News Groups

In [3]:
from sklearn.datasets import fetch_20newsgroups
import pandas as pd

# Para que el ejercicio sea rápido, seleccionaremos solo 4 categorías
categorias = [
    'sci.space', 
    'comp.graphics', 
    'rec.autos', 
    'talk.politics.mideast'
]

print("Descargando el corpus de 20 News Groups...")

# Removemos ('headers', 'footers', 'quotes') para limpiar el texto de metadatos de los correos
newsgroups_train = fetch_20newsgroups(
    subset='train', 
    categories=categorias, 
    remove=('headers', 'footers', 'quotes')
)

# Filtramos documentos vacíos que puedan haber quedado tras la limpieza
corpus = [doc for doc in newsgroups_train.data if doc.strip()]

print(f"¡Listo! Se han cargado {len(corpus)} documentos.")

# Creamos un DataFrame para una visualización más elegante
df_corpus = pd.DataFrame(corpus, columns=['Contenido del Documento'])
df_corpus.head()

Descargando el corpus de 20 News Groups...
¡Listo! Se han cargado 2259 documentos.


,Contenido del Documento
0,\n\tHeed this man's warnings! If you get carb...
1,"# |> >\n# |> >It is NOT a ""terrorist camp"" as ..."
2,\nIf you want info claiming that blacks were b...
3,\n\nExactly. Some of the SPACE:1999 effects re...
4,\n\nFreedom of speech does not mean that other...


Como se puede observar, luego de quitar los encabezados el corpus todavía no es adecuado para generar los embeddings, es por ello que realizaremos un preprocesamiento más a fondo para lograr buenos embeddings.

In [4]:
import re

def limpiar_texto(texto):
    # 1. Reemplazamos saltos de línea y tabulaciones por un espacio
    texto = re.sub(r'[\n\t]', ' ', texto)
    
    # 2. Eliminamos caracteres de citado y otros símbolos molestos (como >, |, #)
    texto = re.sub(r'[>|#]', '', texto)
    
    # 3. Eliminamos espacios múltiples que queden tras los reemplazos
    texto = re.sub(r'\s+', ' ', texto)
    
    # 4. Quitamos espacios al inicio y al final
    return texto.strip()

# Aplicamos la función de limpieza a nuestro corpus original
corpus_limpio = [limpiar_texto(doc) for doc in corpus]

# Filtramos nuevamente por si algún documento quedó completamente vacío tras la limpieza
corpus_limpio = [doc for doc in corpus_limpio if doc]

print(f"Documentos listos para embeddings: {len(corpus_limpio)}")

# Mostramos el DataFrame actualizado
df_corpus_limpio = pd.DataFrame(corpus_limpio, columns=['Contenido del Documento'])
df_corpus_limpio.head()

Documentos listos para embeddings: 2259


,Contenido del Documento
0,Heed this man's warnings! If you get carb clea...
1,"It is NOT a ""terrorist camp"" as you and the Is..."
2,If you want info claiming that blacks were bro...
3,Exactly. Some of the SPACE:1999 effects remain...
4,Freedom of speech does not mean that others ar...


### 2.2 Transformo a embeddings

In [9]:
# Delimitamos el subconjunto a 100 documentos para no exceder los límites de 
# la capa gratuita de la API y agilizar el ejercicio en el Notebook.
textos_a_procesar = corpus_limpio[:100]

print(f"Generando embeddings para {len(textos_a_procesar)} documentos... (Esto puede tomar unos segundos)")

# Llamamos a la API de Gemini para generar los embeddings en lote
respuesta_embeddings = genai.embed_content(
    model="models/gemini-embedding-001",
    content=textos_a_procesar,
    task_type="retrieval_document"
)

# Extraemos la lista de vectores de la respuesta
document_embeddings = respuesta_embeddings['embedding']

print("¡Embeddings generados exitosamente!")
print(f"Total de documentos procesados: {len(document_embeddings)}")
print(f"Dimensiones de cada vector (embedding): {len(document_embeddings[0])}")

# Agregamos los embeddings a nuestro DataFrame para tener todo organizado
df_embeddings = pd.DataFrame({
    'Texto': textos_a_procesar,
    'Embedding': document_embeddings
})

df_embeddings.head()

Generando embeddings para 100 documentos... (Esto puede tomar unos segundos)
¡Embeddings generados exitosamente!
Total de documentos procesados: 100
Dimensiones de cada vector (embedding): 3072


,Texto,Embedding
0,Heed this man's warnings! If you get carb clea...,"[0.005925976, 0.026533803, 0.0088417465, -0.06..."
1,"It is NOT a ""terrorist camp"" as you and the Is...","[-0.015847612, -0.002186583, 0.0043337853, -0...."
2,If you want info claiming that blacks were bro...,"[0.009147593, -0.009456367, 0.023200054, -0.05..."
3,Exactly. Some of the SPACE:1999 effects remain...,"[-0.006141538, 0.010071179, 0.014550169, -0.07..."
4,Freedom of speech does not mean that others ar...,"[0.029638957, 0.022256926, -0.002340698, -0.05..."


### 2.3 Creo una query y hago la búsqueda

In [10]:
import numpy as np

# 1. Definimos nuestra consulta. 
# Como nuestro corpus tiene textos sobre espacio, gráficos, autos y medio oriente, 
# haremos una pregunta relacionada al espacio.
query = "¿Qué información hay sobre las órbitas de los satélites o misiones espaciales?"

print(f"Query: '{query}'")

# 2. Generamos el embedding de la query
respuesta_query = genai.embed_content(
    model="models/gemini-embedding-001",
    content=query,
    task_type="retrieval_query" # Importante
)

query_embedding = respuesta_query['embedding']

# 3. Función matemática para calcular la Similitud del Coseno
def calcular_similitud_coseno(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)

# 4. Calculamos la similitud entre la query y cada documento de nuestro DataFrame
df_embeddings['Similitud'] = df_embeddings['Embedding'].apply(
    lambda doc_emb: calcular_similitud_coseno(query_embedding, doc_emb)
)

print("¡Búsqueda y cálculo de similitudes completado!")

Query: '¿Qué información hay sobre las órbitas de los satélites o misiones espaciales?'
¡Búsqueda y cálculo de similitudes completado!


Obtengo los 5 documentos más similares a mi query

In [ ]:
# Ordenamos el DataFrame por la columna 'Similitud' de forma descendente
top_5_resultados = df_embeddings.sort_values(by='Similitud', ascending=False).head(5)

# Seleccionamos solo las columnas que queremos visualizar
df_resultados_finales = top_5_resultados[['Similitud', 'Texto']]

# Configuramos Pandas para mostrar más contenido de las celdas
pd.set_option('display.max_colwidth', 500)

# Dejamos la variable al final para que Jupyter renderice la tabla
df_resultados_finales

,Similitud,Texto
13,0.708576,"The most current orbital elements from the NORAD two-line element sets are carried on the Celestial BBS, (513) 427-0674, and are updated daily (when possible). Documentation and tracking software are also available on this system. As a service to the satellite user community, the most current elements for the current shuttle mission are provided below. The Celestial BBS may be accessed 24 hours/day at 300, 1200, 2400, 4800, or 9600 bps using 8 data bits, 1 stop bit, no parity. Element sets (..."
14,0.644139,"I gues it is Keesler. The others do not ring the bell but they might be involved as well. Sometime ago Keesler was here at Langley teaching a course on space debris and, if my memory does not fai,l I think there was even a reference to a book on the subject. C.O.Egalon@larc.nasa.gov"
87,0.623060,"Did the Russian spacecraft(s) on the ill-fated Phobos mission a few years ago send back any images of the Martian moon? If so, does anyone know if they're housed at an ftp site? Thanks."
84,0.617777,"COMMERCIAL SPACE NEWS/SPACE TECHNOLOGY INVESTOR NUMBER 22 This is number twenty-two in an irregular series on commercial space activities. The commentaries included are my thoughts on these developments. Sigh... as usual, I've gotten behind in getting this column written. I can only plead the exigency of the current dynamics in the space biz. This column is put together at lunch hour and after the house quiets down at night, so data can quickly build up if there's a lot of other stuff going ..."
36,0.617578,Try FTP-ing at pub-info.jpl.nasa.gov (128.149.6.2) (simple dir-structure) and ames.arc.nasa.gov at /pub/SPACE/GIF and /pub/SPACE/JPEG sorry only 8 bits gifs and jpegs :-( great piccy's though (try the *x.gif files they're semi-huge gif89a files) ^^-watch out gif89a dead ahead!!! Good-luck (good software to be found out-there too) Jurriaan JHWITTEN@CS.RUU.NL


### Conclusiones de la Práctica
A lo largo de este ejercicio, he podido comprender de manera práctica cómo implementar un sistema básico de Recuperación de la Información utilizando la API de Google Gemini. A diferencia de las búsquedas tradicionales exactas o por palabras clave, el uso de embeddings permite que la computadora entienda y busque información basándose en la similitud semántica de los textos.